# First-Pass Data Exploration

This notebook is the readable companion to `2_data/scripts/data_exploration.py`.

The script contains reusable download and coverage logic. This notebook explains why each step is useful, calls that logic, and displays the main results for local inspection.

## Research Motivation

Before building a modeling panel, we need to know whether the candidate target and predictors are feasible.

The key questions are:

1. Which OECD environment-related patent indicators can serve as the target?
2. Which candidate predictors have enough country-year coverage?
3. Which sources may restrict the final sample?
4. Which choices should remain deferred until the literature and metadata are checked?

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]

SCRIPT_DIR = ROOT / "2_data" / "scripts"
PROCESSED_DIR = ROOT / "2_data" / "processed"

sys.path.append(str(SCRIPT_DIR))

from data_exploration import run_exploration

plt.style.use("seaborn-v0_8-whitegrid")

## Load or Regenerate the Exploration Summary

By default, the notebook reads the committed coverage summary from `2_data/processed/data_availability.csv`.

Set `RUN_DOWNLOADS = True` only when you want to refresh the raw downloads and regenerate the summary. Raw files are stored in `2_data/raw/` and are ignored by Git.

In [ ]:
RUN_DOWNLOADS = False
START_YEAR = 1990
END_YEAR = 2024

if RUN_DOWNLOADS:
    summary = run_exploration(start_year=START_YEAR, end_year=END_YEAR)
else:
    summary = pd.read_csv(PROCESSED_DIR / "data_availability.csv")

summary

## Current Result Snapshot

The first-pass exploration found:

1. OECD patent target candidates have strong coverage from 1990 to 2023.
2. `env_patent_share_tech` and `env_patent_share_inventions` each cover 202 countries.
3. `env_patents_per_million` covers 196 countries.
4. OECD EPS covers only 40 countries from 1990 to 2020, so it may restrict the sample.
5. World Bank macro variables have broad coverage, while R&D variables are much thinner.

The final target is still deferred. We should first clarify the OECD metadata distinction between `PT_TECH` and `PT_INV`.

## OECD Target Candidates

The target should represent future environment-related innovation in a way that is comparable across countries and years.

The first-pass candidates are all normalized measures, which helps avoid letting country size dominate the target.

In [ ]:
target_candidates = summary[summary["dataset_id"].eq("oecd_patents_environment")].copy()
target_candidates[[
    "variable",
    "source_variable",
    "non_missing_observations",
    "countries_with_data",
    "first_year",
    "last_year",
]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
target_plot = target_candidates.sort_values("countries_with_data")
ax.barh(target_plot["variable"], target_plot["countries_with_data"], color="#3B6EA8")
ax.set_title("OECD Target Candidate Country Coverage")
ax.set_xlabel("Countries with non-missing data")
ax.set_ylabel("")
for i, value in enumerate(target_plot["countries_with_data"]):
    ax.text(value + 1, i, str(value), va="center")
plt.tight_layout()

## Predictor Coverage

Predictor coverage matters because the final modeling panel will only be as broad as the intersection of target and predictor availability.

A variable can be theoretically attractive but still problematic if it removes too many countries or years.

In [ ]:
predictors = summary[~summary["dataset_id"].eq("oecd_patents_environment")].copy()
predictors = predictors.sort_values(["dataset_id", "non_missing_observations"], ascending=[True, False])
predictors[[
    "dataset_id",
    "variable",
    "source_variable",
    "non_missing_observations",
    "countries_with_data",
    "first_year",
    "last_year",
]]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
predictor_plot = predictors.sort_values("non_missing_observations")
colors = predictor_plot["dataset_id"].map({
    "world_bank_wdi": "#4C9A6A",
    "oecd_eps": "#C46A3A",
}).fillna("#666666")
ax.barh(predictor_plot["variable"], predictor_plot["non_missing_observations"], color=colors)
ax.set_title("Candidate Predictor Coverage")
ax.set_xlabel("Non-missing country-year observations")
ax.set_ylabel("")
plt.tight_layout()

## Year Coverage

The target candidates begin in 1990, but several predictors start later or end earlier. This matters for the eventual lag structure.

A predictor that begins in 1996, for example, cannot support a model that predicts 1991 innovation with a `t-1` lag.

In [ ]:
coverage = summary.sort_values(["first_year", "last_year", "variable"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))
for y, row in coverage.iterrows():
    ax.plot([row["first_year"], row["last_year"]], [y, y], linewidth=4)

ax.set_yticks(range(len(coverage)))
ax.set_yticklabels(coverage["variable"])
ax.set_title("Observed Year Coverage by Candidate Variable")
ax.set_xlabel("Year")
ax.set_ylabel("")
plt.tight_layout()

## Interpretation for the Next Research Step

The data are feasible enough to continue, but the project should not jump to modeling yet.

Recommended next steps:

1. Read the OECD patent metadata to decide between `PT_TECH`, `PT_INV`, and `INV_PS`.
2. Decide whether the project prioritizes broad country coverage or stronger policy variables.
3. Treat EPS as valuable but sample-restricting.
4. Be cautious with R&D variables because they have strong theoretical value but thinner coverage.
5. Record the final target decision in `0_organization/decision_log.md`.